# Pruning

This notebook investigates the utility of using [Fil-Finder](https://fil-finder.readthedocs.io/en/latest/index.html) to prune skeletons, although on reading it appears that
it may be useful for skeletonising at the same time.

In [ ]:
from pathlib import Path

# import fil_finder.FilFinder2D as ff
from fil_finder import FilFinder2D as ff
import numpy as np
import matplotlib.pyplot as plt

from topostats.filters import Filters
from topostats.grains import Grains
from topostats.grainstats import GrainStats
from topostats.io import find_files, read_yaml, write_yaml, LoadScans
from topostats.logs.logs import setup_logger, LOGGER_NAME
from topostats.plottingfuncs import Images
from topostats.tracing.dnatracing import dnaTrace, prep_arrays
from topostats.utils import update_config

In [ ]:
BASE_DIR = Path().cwd()
RESOURCES = BASE_DIR.parent / "tests" / "resources"
FILE_EXT = ".spm"
image_files = find_files(base_dir=BASE_DIR.parent / "tests", file_ext=FILE_EXT)
config = read_yaml(BASE_DIR.parent / "topostats" / "default_config.yaml")
loading_config = config["loading"]
filter_config = config["filter"]
filter_config.pop("run")
grain_config = config["grains"]
grain_config.pop("run")
grainstats_config = config["grainstats"]
grainstats_config.pop("run")
dnatracing_config = config["dnatracing"]
dnatracing_config.pop("run")
plotting_config = config["plotting"]
plotting_config.pop("run")
plotting_config["image_set"] = "all"

# Load and Detect Grains in `minicircle.spm`

In [ ]:
all_scan_data = LoadScans(image_files, **config["loading"])
all_scan_data.get_data()
filtered_image = Filters(
    image=all_scan_data.img_dict["minicircle"]["image"],
    filename=all_scan_data.img_dict["minicircle"]["img_path"],
    pixel_to_nm_scaling=all_scan_data.img_dict["minicircle"]["px_2_nm"],
    **filter_config,
)
filtered_image.filter_image()

In [ ]:
# grain_config["threshold_absolute"] = {"below": -2.0, "above": 1.2}  # Change both the below and above threshold
grains = Grains(
    image=filtered_image.images["zero_average_background"],
    filename=filtered_image.filename,
    pixel_to_nm_scaling=filtered_image.pixel_to_nm_scaling,
    **grain_config,
)
grains.find_grains()

In [ ]:
print(grains.directions["above"].keys())
fig, ax = Images(
    data=grains.directions["above"]["coloured_regions"],
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
fig

In [ ]:
binary_mask = np.where(grains.directions["above"]["coloured_regions"] == 0, 0, 1)
fig, ax = Images(
    data=binary_mask,
    filename=filtered_image.filename,
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
fig

# Skan

Found the promising [Skan](https://skeleton-analysis.org/stable/index.html) package via a
[thread](https://stackoverflow.com/questions/67244206/using-skimage-and-fil-finder-in-measuring-the-length-of-skeleton)
so lets have a play.

In [ ]:
from skimage.measure import label

In [ ]:
labelled_grains_mask = label(grains.directions["above"]["coloured_regions"])

In [ ]:
cropped_images, cropped_masks = prep_arrays(
    image=filtered_image.images["zero_average_background"],
    labelled_grains_mask=grains.directions["above"]["labelled_regions_02"],
    pad_width=2,
)
print(f"cropped_images : {len(cropped_images)}")
print(f"cropped_masks : {len(cropped_masks)}")
plt.imshow(cropped_images[0])

In [ ]:
from skimage import morphology

skeletons = [morphology.skeletonize(grain) for grain in cropped_masks]
fig, ax = plt.subplots()
ax.imshow(skeletons[0])

In [ ]:
from skan import draw

fig, ax = plt.subplots()
# skeletons[0][0]
draw.overlay_skeleton_2d(cropped_images[0], skeletons[0], dilate=1, axes=ax)

In [ ]:
from skan.csr import skeleton_to_csgraph

spacing = all_scan_data.img_dict["minicircle"]["px_2_nm"]
pixel_graph, coordinates = skeleton_to_csgraph(skeletons[0], spacing=spacing)

In [ ]:
fig, ax = plt.subplots(figsize=(40, 24))
draw.overlay_skeleton_networkx(pixel_graph, np.transpose(coordinates), image=cropped_images[0], axis=ax)

In [ ]:
from skan import Skeleton, summarize

skan_skel = Skeleton(skeletons[0], spacing=spacing)  # , value_is_height=True)
branch_data = summarize(skan_skel)
branch_data["Branch Type"] = branch_data["branch-type"].map(
    {0: "end-to-end (isolated branch)", 1: "junction-to-endpoint", 2: "junction-to-junction", 3: "isolated cycle"}
)
print(f"Columns :\n{branch_data.columns}")

In [ ]:
branch_data.hist(column="branch-distance", by="Branch Type", bins=80)

In [ ]:
draw.overlay_euclidean_skeleton_2d(cropped_images[0], branch_data, skeleton_color_source="branch-type")

In [ ]:
branches = branch_data[branch_data["branch-type"] < 2]
branches

In [ ]:
pruned_skeleton = skan_skel.prune_paths(branches.index)

In [ ]:
plt.imshow(skeletons[0])

In [ ]:
binary_pruned = np.where(pruned_skeleton.path_label_image() == 0, 0, 1)
plt.imshow(binary_pruned)

## Second Iteration



In [ ]:
pruned_skeleton = Skeleton(
    binary_pruned, source_image=cropped_images[0], keep_images=True, spacing=spacing, value_is_height=False
)
pruned_skeleton_summary = summarize(pruned_skeleton)

In [ ]:
second_pruned = pruned_skeleton.prune_paths(pruned_skeleton_summary[pruned_skeleton_summary["branch-type"] < 2].index)
binary_pruned2 = np.where(second_pruned.path_label_image() == 0, 0, 1)
plt.imshow(binary_pruned2)

## Third Iteration

In [ ]:
pruned_skeleton = Skeleton(
    binary_pruned2, source_image=cropped_images[0], keep_images=True, spacing=spacing, value_is_height=False
)
pruned_skeleton_summary = summarize(pruned_skeleton)
print(pruned_skeleton_summary)
third_pruned = pruned_skeleton.prune_paths(pruned_skeleton_summary[pruned_skeleton_summary["branch-type"] < 2].index)
binary_pruned3 = np.where(third_pruned.path_label_image() == 0, 0, 1)
plt.imshow(binary_pruned3)

In [ ]:
import nu

In [ ]:
def recursively_prune_paths(
    self, remove_branch_type: int = 1, cleanup: bool = False, skeletonize_method: str = "zhang", spacing=1
) -> "Skeleton":
    image_cp = np.copy(self.skeleton_image)

    # Summarise the skeleton we need to know what each branchtype is
    # Is there a more direct way of getting these?
    summary = summarize(image_cp)

    while remove_branch_type in summary["branch-type"].unique():
        # Extract indices of of the fiven branch type
        indices = summary.loc[summary["branch-type"] == remove_branch_type].index

        for i in indices:
            pixel_ids_to_wipe = self.path(i)
            junctions = self.degrees[pixel_ids_to_wipe] > 2
            pixel_ids_to_wipe = pixel_ids_to_wipe[~junctions]
            coords_to_wipe = self.coordinates[pixel_ids_to_wipe]
            coords_idxs = tuple(np.round(coords_to_wipe).astype(int).T)
            image_cp[coords_idxs] = 0
        # Optionally cleanup:
        new_skeleton = morphology.skeletonize(image_cp.astype(bool)) * image_cp if cleanup else image_cp.astype(bool)
        summary = summarize(new_skeleton)
    return Skeleton(
        new_skeleton,
        spacing=spacing,
    )


recursively_prune_paths(skeletons[0], remove_branch_type=1, cleanup=False, spacing=1)

In [ ]:
pruned = branch_data[branch_data["branch-type"] >= 2][["image-coord-src-0", "image-coord-src-1"]]
pruned_skeleton = np.zeros(cropped_images[0].shape)
pruned_skeleton[pruned["image-coord-src-0"], pruned["image-coord-src-1"]] = 1
fig, ax = plt.subplots()
plt.imshow(pruned_skeleton)

In [ ]:
from skan.csr import Skeleton
from skan.csr import summarize

skan_skel = Skeleton(
    skeletons[0], source_image=cropped_images[0], keep_images=True, spacing=spacing, value_is_height=False
)
summarize(skan_skel)
for index, row in branch_data.iterrows():
    print(f"Index   : {index}")
    print(f"Node ID : {branch_data.loc[index,'node-id-src']}")
    # print(f"path    : {skan_skel.path(branch_data.loc[index,'node-id-src'])}")
skan_skel.prune_paths(branch_data["node-id-src"])

In [ ]:
branches =

In [ ]:
CIRCULAR_MASK = np.load(RESOURCES / "dnatracing_mask_circular.npy")
fig, ax = plt.subplots()
plt.imshow(CIRCULAR_MASK)

In [ ]:
CIRCLE = np.asarray(
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ]
)
circle_pixel_graph, circle_coordinates = skeleton_to_csgraph(CIRCLE, spacing=spacing)

In [ ]:
fig, ax = plt.subplots()
draw.overlay_skeleton_networkx(circle_pixel_graph, np.transpose(circle_coordinates), axis=ax)

In [ ]:
circle_branch_data = summarize(Skeleton(CIRCLE, spacing=spacing))
circle_branch_data["Branch Type"] = circle_branch_data["branch-type"].map(
    {0: "end-to-end (isolated branch)", 1: "junction-to-endpoint", 2: "junction-to-junction", 3: "isolated cycle"}
)
print(f"Columns :\n{circle_branch_data.columns}")

In [ ]:
circle_branch_data

In [ ]:
circle_branch_data.hist(column="branch-distance", by="Branch Type", bins=80)

In [ ]:
draw.overlay_euclidean_skeleton_2d(CIRCLE, circle_branch_data, skeleton_color_source="branch-type")